# Projet - SQL : Interroger une base de réservations de vols



<div align="center">
<img src="./image.png" alt="ouvrir dans VSCode pour voir l'image" width="800"/>
</div>

Enfin nous y sommes ! bientot c'est Vacances de noel ! :)

Tu viens de finir ta dernière journée de travail, et direction de kinshasa ! Il fait 32 degrés, tu as ta valise à la main et ton vol pour libreville est dans 3 heures. En attendant, autant pratiquer un peu de SQL (Pour réussir, il faut toujours pratiquer !)

Le but de ce TP est de te glisser dans la peau d'un analyste qui travaille pour l'aéroport. Sous tes yeux, le système de réservations : des passagers, des vols, des réservations. Ta mission sera de poser des questions à ces données et d'en tirer des réponses précises :

- Quels vols décollent de kananga cette semaine ?
- Quelles destinations rapportent le plus d'argent ?
- Qui embarque sur le vol de New york ?

À la fin du projet, tu sauras transformer n'importe quelle question de ce genre en une requête qui te donne la réponse en quelques millisecondes. Et bonne nouvelle, pas besoin d'être développeur ici pour y arriver : on part de zéro et chaque exercice vérifie tout seul que ta réponse est la bonne.

### L'outil du jour : SQL

Aujourd'hui, nous sommes entourés de bases de données relationnelles : elles font tourner les sites web, les applications mobiles, les banques, les systèmes de gestion et bien sûr les compagnies aériennes. À chaque fois que tu réserves un billet d'avion, une base de données enregistre quelque part qui tu es, quel vol tu prends et quel siège tu occupes.

SQL (Structured Query Language) est le langage standard pour dialoguer avec ces bases : on écrit une question dans un format précis, qu'on appelle une requête, et la base nous renvoie la réponse. Pour un Data Analyst, un Data Engineer ou toute autre personne travaillant avec de la data, c'est sans doute l'outil le plus utilisé au quotidien, et il n'a presque pas changé depuis 50 ans. 

Ici on utilisera SQLite, une version légère qui tient dans un simple fichier (ou même dans la mémoire de l'ordinateur), parfaite pour apprendre. Rien de très exotique comme technologie, mais les concepts sont exactement les mêmes que sur PostgreSQL ou MySQL, les moteurs qu'on retrouve en entreprise.

Au programme : les requêtes de base, les agrégations, les jointures et les sous-requêtes. Autrement dit, les briques qu'on utilise vraiment tous les jours dans les métiers de la data.

# Avant tout : creation de l'environnement virtuel et installation de librairies 

## ⚙️ Étape 0 : Préparation de l'environnement

### 0.1 Vérifier que Python est installé

Ouvre un terminal (PowerShell ou CMD sous Windows, Terminal sous macOS) et tape :

**Windows**

```bash
python --version
```

**macOS / Linux**

```bash
python3 --version
```

Résultat attendu : un numéro de version, par exemple `Python 3.12.4`. Il faut **Python 3.9 ou plus**.

Si la commande n'est pas reconnue, installe Python depuis [python.org/downloads](https://www.python.org/downloads/). Sous Windows, **coche la case « Add Python to PATH »** pendant l'installation.

Vérifie aussi `pip` :

```bash
pip --version
```

### 0.2 Créer le dossier du projet

```bash
mkdir base-de-reservation
cd base-de-reservation
```

### 0.3 Créer l'environnement virtuel

L'environnement virtuel isole les bibliothèques du projet, pour ne pas mélanger les versions avec d'autres projets.

**Windows**

```bash
python -m venv venv
```

**macOS / Linux**

```bash
python3 -m venv venv
```

Un dossier `venv/` est créé dans le projet.

### 0.4 Activer l'environnement virtuel

**Windows (CMD)**

```bash
venv\Scripts\activate
```

**Windows (PowerShell)**

```bash
venv\Scripts\Activate.ps1
```

Si PowerShell bloque le script, lance une fois :

```bash
Set-ExecutionPolicy -Scope CurrentUser RemoteSigned
```

**macOS / Linux**

```bash
source venv/bin/activate
```

Quand c'est activé, `(venv)` apparaît au début de la ligne du terminal. Pour quitter l'environnement : `deactivate`.

### 0.5 Installer les bibliothèques

```bash
pip install pandas 
```

Sauvegarde les versions utilisées pour pouvoir refaire le projet à l'identique :

```bash
pip freeze > requirements.txt
```

Plus tard, n'importe qui pourra reconstruire l'environnement avec :

```bash
pip install -r requirements.txt
```


In [2]:
import sqlite3
import pandas as pd

# Partie 1 Création de la base de données

Avant de pouvoir interroger quoi que ce soit, il nous faut... des données ! Dans la vraie vie, la base de réservations existe déjà : c'est le système de la compagnie aérienne qui la remplit à chaque billet vendu. Ici, on va la fabriquer nous-mêmes, en miniature : 15 passagers, 15 vols et une vingtaine de réservations. C'est l'occasion parfaite pour découvrir comment une base de données se construit, avant d'apprendre à l'interroger.

Si vous n'êtes pas familier avec le langage SQL, je vous invite à consulter ce cours en ligne gratuit et assez simple. Il vous aidera à faire vos premiers pas et à réaliser la mission de ce projet.

https://www.w3schools.com/sql/sql_intro.asp


Notre base contient trois tables :

```txt
passengers (id, first_name, last_name, email, nationality)
flights    (id, flight_number, origin, destination, departure_time, arrival_time, aircraft, capacity, price_eur)
bookings   (id, passenger_id, flight_id, booking_date, seat_class, seat_number, status)
```

<div align="center">
<img src="./base_de_données.png" alt="ouvrir dans VSCode pour voir l'image" width="800"/>
</div>

La table `bookings` fait le lien entre les deux autres : chaque réservation pointe vers un passager (`passenger_id`) et vers un vol (`flight_id`). C'est exactement ça, une base relationnelle : des tables reliées entre elles par des identifiants.

On commence par ouvrir une connexion vers une base SQLite en mémoire. "En mémoire" signifie que la base vit dans la RAM le temps de la session : aucun fichier n'est créé, et la base est recréée à chaque exécution du notebook. C'est voulu, ça évite les surprises si tu relances les cellules dans le désordre.

In [3]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

print("Connexion à la base établie !")

Connexion à la base établie !


### Table 1 : les passagers

On crée la première table ensemble, et on en profite pour décortiquer la syntaxe de `CREATE TABLE`. Chaque ligne déclare une colonne avec son nom, son type, et éventuellement une contrainte :

- `id INTEGER PRIMARY KEY` : un entier qui identifie chaque passager de façon unique. C'est la clé primaire, celle vers laquelle les autres tables pourront pointer.
- `first_name TEXT NOT NULL` et `last_name TEXT NOT NULL` : du texte, avec la contrainte `NOT NULL` qui interdit d'enregistrer un passager sans prénom ou sans nom.
- `email TEXT UNIQUE` : la contrainte `UNIQUE` garantit que deux passagers ne peuvent pas partager la même adresse email.
- `nationality TEXT` : du texte, sans contrainte particulière, la colonne peut rester vide.

Une fois la table créée, on la remplit avec `INSERT INTO` via `executemany`, qui insère toutes les lignes d'un coup. Exécute la cellule, elle vérifie toute seule que tout est en place.

Pour commencer, nous allons écrire la commande permettant de créer une table `passagers`

In [4]:
cursor.execute("""
CREATE TABLE passengers (
    id          INTEGER PRIMARY KEY,
    first_name  TEXT NOT NULL,
    last_name   TEXT NOT NULL,
    email       TEXT UNIQUE,
    nationality TEXT
);
""")

Maintenant que c'est fait, nous pouvons remplir cette table avec les données suivantes : 

In [5]:
passengers_data = [
    (1, 'Lucas',   'Moreau',   'lucas.moreau@email.fr',    'Congolaise'),
    (2, 'Emma',    'Dubois',   'emma.dubois@email.fr',     'Congolaise'),
    (3, 'Noah',    'Leroy',    'noah.leroy@email.fr',      'Congolaise'),
    (4, 'Sofia',   'Rossi',    'sofia.rossi@email.it',     'ivorienne'),
    (5, 'Liam',    'Smith',    'liam.smith@email.uk',      'ghanaise'),
    (6, 'Mia',     'Müller',   'mia.muller@email.de',      'rwandaise'),
    (7, 'Hugo',    'Bernard',  'hugo.bernard@email.fr',    'Congolaise'),
    (8, 'Camille', 'Thomas',   'camille.thomas@email.fr',  'Congolaise'),
    (9, 'Alice',   'Martin',   'alice.martin@email.fr',    'Congolaise'),
    (10,'Carlos',  'Garcia',   'carlos.garcia@email.es',   'gabonaise'),
    (11,'Jade',    'Petit',    'jade.petit@email.fr',      'Congolaise'),
    (12,'Omar',    'Hassan',   'omar.hassan@email.ma',     'Marocaine'),
    (13,'Elena',   'Popescu',  'elena.popescu@email.ro',   'nigériane'),
    (14,'Antoine', 'Garnier',  'antoine.garnier@email.fr', 'Congolaise'),
    (15,'Yako',    'Tanaka',   'yuki.tanaka@email.jp',     'tanzaine'),
]

In [6]:
cursor.executemany("INSERT INTO passengers VALUES (?,?,?,?,?)", passengers_data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM passengers")
count = cursor.fetchone()[0]
assert count == 15, f"La table passengers devrait contenir 15 passagers, elle en a {count}"
print(f"Table passengers créée et remplie avec {count} passagers !")

Table passengers créée et remplie avec 15 passagers !


**Remarque** : Si tu réexécutes les cellules de code ci-dessus, tu rencontreras une erreur, car l'instruction de création de la table passengers entrera en conflit avec le fait que cette table existe déjà.

Sois donc vigilant lorsque tu relances ces cellules de code ! (Dans le pire des cas, on peut toujours redémarrer le Kernel)

### Table 2 : les vols (à toi de jouer)

Maintenant que tu as vu la syntaxe, à ton tour de créer la table `flights`.

Celle-ci recense les vols avec leurs numéros, leurs horaires, l'avion utilisé et le prix du billet.

Complète la commande `CREATE TABLE` entre les balises avec les colonnes suivantes :

| Colonne | Type | Contrainte |
|---|---|---|
| `id` | INTEGER | PRIMARY KEY |
| `flight_number` | TEXT | NOT NULL |
| `origin` | TEXT | NOT NULL |
| `destination` | TEXT | NOT NULL |
| `departure_time` | TEXT | NOT NULL |
| `arrival_time` | TEXT | NOT NULL |
| `aircraft` | TEXT | |
| `capacity` | INTEGER | |
| `price_eur` | REAL | |

N'oublie pas la virgule entre chaque colonne (mais pas après la dernière). Les données sont déjà prêtes dans la cellule, ne les modifie pas : les exercices de la suite du projet comptent dessus, et la cellule vérifie automatiquement ton schéma.

In [9]:
cursor.execute("""
-- ### START CODE HERE ###
              
-- ### END CODE HERE ###
""")

Une fois les lignes de code ci-dessus complétées, on peut remplir notre nouvelle table avec les données suivantes :

In [13]:
flights_data = [
    (1,  'AF1234', 'kinshasa',      'Paris CDG',    '2026-07-01 07:00', '2026-07-01 08:30', 'Airbus A320', 180, 89.0),
    (2,  'AF1235', 'lubumbashi', 'Nice',          '2026-07-01 19:00', '2026-07-01 20:30', 'Airbus A320', 180, 95.0),
    (3,  'EZ4501', 'kinshasa',      'Londres',       '2026-07-02 06:30', '2026-07-02 08:15', 'Airbus A319', 156, 120.0),
    (4,  'EZ4502', 'kinshasa',      'Amsterdam',     '2026-07-02 09:00', '2026-07-02 11:30', 'Airbus A319', 156, 135.0),
    (5,  'VY6010', 'kinshasa',      'Barcelone',     '2026-07-03 11:00', '2026-07-03 12:45', 'Boeing 737',  189, 78.0),
    (6,  'AF5501', 'kinshasa',      'New York JFK',  '2026-07-04 13:00', '2026-07-04 22:30', 'Boeing 777',  350, 650.0),
    (7,  'LH2201', 'kinshasa',      'Francfort',     '2026-07-05 08:00', '2026-07-05 09:45', 'Airbus A320', 180, 112.0),
    (8,  'IB3301', 'kinshasa',      'Madrid',        '2026-07-05 14:00', '2026-07-05 16:00', 'Boeing 737',  189, 99.0),
    (9,  'AF1236', 'kinshasa',      'Paris CDG',     '2026-07-06 18:00', '2026-07-06 19:30', 'Airbus A320', 180, 105.0),
    (10,'U24441', 'kinshasa',      'Athènes',       '2026-07-07 10:00', '2026-07-07 13:30', 'Boeing 737',  189, 145.0),
    (11, 'FR7701', 'kinshasa',      'Dublin',        '2026-07-08 07:30', '2026-07-08 10:00', 'Boeing 737',  189, 89.0),
    (12, 'AF1237', 'kinshasa',      'Paris CDG',     '2026-07-08 20:00', '2026-07-08 21:30', 'Airbus A321', 220, 88.0),
    (13, 'EZ4503', 'kinshasa',      'Berlin',        '2026-07-09 12:00', '2026-07-09 14:15', 'Airbus A319', 156, 118.0),
    (14, 'AF5502', 'kinshasa',      'New York JFK',  '2026-07-10 13:00', '2026-07-10 22:30', 'Boeing 777',  350, 680.0),
    (15, 'TK8801', 'kinshasa',      'Istanbul',      '2026-07-11 09:00', '2026-07-11 12:30', 'Boeing 737',  189, 160.0),
]

In [ ]:
cursor.executemany("INSERT INTO flights VALUES (?,?,?,?,?,?,?,?,?)", flights_data)
conn.commit()

cursor.execute("PRAGMA table_info(flights)")
columns = cursor.fetchall()
schema = {col[1]: col[2] for col in columns}
expected_schema = {
    'id': 'INTEGER', 'flight_number': 'TEXT', 'origin': 'TEXT', 'destination': 'TEXT',
    'departure_time': 'TEXT', 'arrival_time': 'TEXT', 'aircraft': 'TEXT',
    'capacity': 'INTEGER', 'price_eur': 'REAL',
}
assert schema == expected_schema, f"Le schéma ne correspond pas. Attendu : {expected_schema}, obtenu : {schema}"

not_null_columns = {col[1] for col in columns if col[3] == 1}
expected_not_null = {'flight_number', 'origin', 'destination', 'departure_time', 'arrival_time'}
assert not_null_columns == expected_not_null, \
    f"Les colonnes NOT NULL attendues sont {expected_not_null}, tu as : {not_null_columns}"

cursor.execute("SELECT COUNT(*) FROM flights")
count = cursor.fetchone()[0]
assert count == 15, f"La table flights devrait contenir 15 vols, elle en a {count}"
print(f"Table flights créée et remplie avec {count} vols, exercice validé !")

### Table 3 : les réservations (à toi de jouer)

Dernière table, et pas des moindres : c'est elle qui relie les passagers aux vols. Deux nouveautés par rapport aux précédentes :

- `REFERENCES` : la colonne `passenger_id` pointe vers l'`id` de la table `passengers`, et `flight_id` vers l'`id` de la table `flights`. C'est ce qu'on appelle une clé étrangère, le ciment des bases relationnelles.
- `CHECK` : cette contrainte restreint les valeurs autorisées dans une colonne. Par exemple `CHECK(seat_class IN ('economy', 'business', 'first'))` refusera toute valeur autre que ces trois classes de siège.

Voici les colonnes à déclarer :

| Colonne | Type | Contrainte |
|---|---|---|
| `id` | INTEGER | PRIMARY KEY |
| `passenger_id` | INTEGER | REFERENCES passengers(id) |
| `flight_id` | INTEGER | REFERENCES flights(id) |
| `booking_date` | TEXT | |
| `seat_class` | TEXT | CHECK parmi `'economy'`, `'business'`, `'first'` |
| `seat_number` | TEXT | |
| `status` | TEXT | CHECK parmi `'confirmed'`, `'cancelled'`, `'pending'` |

Comme avant, ne modifie pas les données : la cellule vérifie le schéma, et teste même que ta contrainte `CHECK` refuse bien une valeur interdite.

In [ ]:
cursor.execute("""
-- ### START CODE HERE ###
               

-- ### END CODE HERE ###
""")

une fois les lignes de code ci-dessus complétée, on peut remplir notre nouvelle table avec les données suivantes :

In [ ]:
bookings_data = [
    (1,  1,  1,  '2026-06-01', 'economy',  '14A', 'confirmed'),
    (2,  2,  1,  '2026-06-02', 'business', '2C',  'confirmed'),
    (4,  4,  5,  '2026-06-05', 'economy',  '18C', 'confirmed'),
    (5,  5,  3,  '2026-06-06', 'business', '1A',  'confirmed'),
    (6,  6,  7,  '2026-06-07', 'economy',  '31D', 'confirmed'),
    (7,  7,  6,  '2026-06-08', 'first',    '1A',  'confirmed'),
    (8,  8,  6,  '2026-06-09', 'business', '3B',  'confirmed'),
    (9,  9,  9,  '2026-06-10', 'economy',  '25A', 'confirmed'),
    (10, 10, 5,  '2026-06-11', 'economy',  '17B', 'cancelled'),
    (11, 11, 10, '2026-06-12', 'economy',  '8C',  'confirmed'),
    (12, 12, 15, '2026-06-13', 'economy',  '11A', 'confirmed'),
    (13, 13, 4,  '2026-06-14', 'economy',  '29B', 'pending'),
    (14, 14, 2,  '2026-06-15', 'business', '4A',  'confirmed'),
    (15, 15, 6,  '2026-06-16', 'first',    '2A',  'confirmed'),
    (16, 1,  9,  '2026-06-17', 'economy',  '33C', 'confirmed'),
    (17, 2,  10, '2026-06-18', 'economy',  '15A', 'cancelled'),
    (18, 5,  6,  '2026-06-19', 'business', '5B',  'confirmed'),
    (19, 7,  14, '2026-06-20', 'first',    '1B',  'confirmed'),
    (20, 9,  11, '2026-06-21', 'economy',  '22D', 'confirmed'),
]

In [ ]:
cursor.executemany("INSERT INTO bookings VALUES (?,?,?,?,?,?,?)", bookings_data)
conn.commit()

cursor.execute("PRAGMA table_info(bookings)")
schema = {col[1]: col[2] for col in cursor.fetchall()}
expected_schema = {
    'id': 'INTEGER', 'passenger_id': 'INTEGER', 'flight_id': 'INTEGER',
    'booking_date': 'TEXT', 'seat_class': 'TEXT', 'seat_number': 'TEXT', 'status': 'TEXT',
}
assert schema == expected_schema, f"Le schéma ne correspond pas. Attendu : {expected_schema}, obtenu : {schema}"

try:
    cursor.execute("INSERT INTO bookings VALUES (999, 1, 1, '2026-06-30', 'premium', '1A', 'confirmed')")
    raise AssertionError("La contrainte CHECK sur seat_class ne fonctionne pas : 'premium' a été accepté")
except sqlite3.IntegrityError:
    pass

cursor.execute("SELECT COUNT(*) FROM bookings")
count = cursor.fetchone()[0]
assert count == 19, f"La table bookings devrait contenir 19 réservations, elle en a {count}"

print("Base de données complète, exercice validé !")
print(f"  {len(passengers_data)} passagers")
print(f"  {len(flights_data)} vols")
print(f"  {len(bookings_data)} réservations")

## Petite fonction utilitaire

Dernière étape de préparation. Pour interroger la base, on pourrait appeler `pd.read_sql_query(...)` à chaque fois, mais c'est un peu long à écrire. On crée donc une petite fonction `query` qui prend une requête SQL (sous forme de texte) et renvoie directement le résultat sous forme de DataFrame pandas, ce fameux tableau bien lisible. C'est elle que tu utiliseras dans tous les exercices.

La cellule affiche aussi les 5 premières lignes de chaque table, histoire de voir enfin à quoi ressemblent nos données.

In [ ]:
def query(sql):
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("=== PASSENGERS ===")
display(query("SELECT * FROM passengers LIMIT 5"))
print("=== FLIGHTS ===")
display(query("SELECT * FROM flights LIMIT 5"))
print("=== BOOKINGS ===")
display(query("SELECT * FROM bookings LIMIT 5"))

---

## Partie 2 - Requêtes de base

Maintenant que nous avons créé les différentes tables de notre bases de données, nous allons désormais jouer avec elles en écrivant quelques requêtes simples.

### Requête #1: Lister tous les vols au départ de Nice

Récupère le numéro de vol (`flight_number`), la destination, l'heure de départ et le prix, pour tous les vols dont l'origine est `'Nice'`. Trie les résultats par heure de départ.

In [ ]:
result = query("""
-- ### START CODE HERE ###
               SELECT flight_number, destination, departure_time, price_eur
                FROM flights
                WHERE origin = 'Nice'

-- ### END CODE HERE ###
""")

display(result)

assert len(result) == 13, f"Tu devrais avoir 13 vols au départ de Nice, tu en as {len(result)}"
assert list(result.columns) == ['flight_number', 'destination', 'departure_time', 'price_eur'], \
    "Vérifie les colonnes sélectionnées"
print("Exercice validé !")

### Requête #2: Identifier Les vols pas chers

Récupère tous les vols dont le prix est inférieur à 100 euros. Affiche le numéro de vol, la destination et le prix. Trie par prix croissant.

In [ ]:
result = query("""
-- ### START CODE HERE ###
                SELECT flight_number, destination, departure_time, price_eur
                 FROM flights
                 WHERE price_eur < 100
                 ORDER BY price_eur ASC

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 6, f"Tu devrais avoir 6 vols sous 100 euros, tu en as {len(result)}"
assert result['price_eur'].is_monotonic_increasing, "Les résultats doivent être triés par prix croissant"
print("Exercice validé !")

### Requête #3: Rechercher des noms de passager

Récupère toutes les infos de la passagère dont le nom de famille est `'Tanaka'`.

In [ ]:
result = query("""
-- ### START CODE HERE ###
               SELECT *
                FROM passengers
                WHERE last_name = 'Tanaka'

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 1
assert result.iloc[0]['first_name'] == 'Yako'
print("Exercice validé !")